## Imports

In [ ]:
import pandas as pd
import numpy as np

## Load Raw Data


In [ ]:
# Load the raw data files

clinical = pd.read_csv(f"{BASE_PATH}/data_raw/clinical.txt", sep="\t", comment="#")
rna = pd.read_csv(f"{BASE_PATH}/data_raw/rna_zscores.txt", sep="\t")
mut = pd.read_csv(f"{BASE_PATH}/data_raw/mutations.txt", sep="\t")

/tmp/ipython-input-150/2619754424.py:5: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  mut = pd.read_csv(f"{BASE_PATH}/data_raw/mutations.txt", sep="\t")


## Clean Clinical

In [ ]:
clinical = clinical.rename(columns={
    "PATIENT_ID": "Patient_ID",
    "OS_STATUS": "OS_STATUS",
    "OS_MONTHS": "OS_MONTHS"
})

clinical["event"] = clinical["OS_STATUS"].apply(lambda x: 1 if "DECEASED" in str(x) else 0)
clinical["time"] = pd.to_numeric(clinical["OS_MONTHS"], errors="coerce")

clinical = clinical[["Patient_ID", "time", "event"]].dropna()
clinical.head()

,Patient_ID,time,event
0,TCGA-3C-AAAU,133.050597,0
1,TCGA-3C-AALI,131.669790,0
2,TCGA-3C-AALJ,48.459743,0
3,TCGA-3C-AALK,47.604958,0
4,TCGA-4H-AAAK,11.440971,0


## Process RNA

In [ ]:
# 1. Create a new DataFrame with patient IDs and gene expression values
rna_cleaned_data = rna.iloc[1:, 4:].copy() # Select data: rows are patients, columns are genes
rna_cleaned_data.columns = rna.columns[4:] # Assign correct gene symbols as column names
rna_cleaned_data.insert(0, 'Patient_ID', rna.iloc[1:, 0].values) # Insert Patient_ID as the first column

# Standardize Patient_ID: remove '-XX' suffix if present to match clinical data
rna_cleaned_data['Patient_ID'] = rna_cleaned_data['Patient_ID'].str.replace(r'-\d{2}$', '', regex=True)

# Update the 'rna' DataFrame to this cleaned version for subsequent steps
rna = rna_cleaned_data

# --- Column Name Cleaning and Uniqueness Enforcement ---
# Remove any columns whose names are NaN. (Should not be necessary with the new construction but as a safeguard)
rna = rna.loc[:, rna.columns.notna()]

# Ensure all remaining column names are unique. If duplicates exist, append a suffix.
cols = pd.Series(rna.columns)
for dup_col_name in cols[cols.duplicated()].unique():
    mask = cols == dup_col_name
    # Append a unique suffix to duplicated column names.
    cols[mask] = [f'{dup_col_name}_{i}' for i in range(mask.sum())]
rna.columns = cols
# -----------------------------------------------------

# Optional: variance filtering
# Calculate variance only on numeric columns to avoid errors with 'Patient_ID' or other non-numeric columns.
numeric_cols = rna.select_dtypes(include=np.number).columns.drop('Patient_ID', errors='ignore')
gene_variance = rna[numeric_cols].var()

# Drop any genes with NaN variance (e.g., if a gene column was entirely NaNs after cleaning)
gene_variance.dropna(inplace=True)

top_genes = gene_variance.sort_values(ascending=False).head(2000).index

# Select 'Patient_ID' and the top filtered genes for the final DataFrame
rna_filtered = rna[["Patient_ID"] + list(top_genes)]

# Save the filtered data to a Parquet file
rna_filtered.to_parquet(f"{BASE_PATH}/data_processed/expression_filtered.parquet")

## Merge Multi-Omics

In [ ]:
data = pd.merge(clinical, rna_filtered,
                left_on="Patient_ID",
                right_on="Patient_ID")

data.to_parquet(f"{BASE_PATH}/data_processed/multiomics_merged.parquet")
print("Merged data head:")
display(data.head())

Merged data head:


,Patient_ID,time,event,CEACAM18,KRTAP12-4,RPS4Y2,DEFB134,TTTY4C,TMEM189-UBE2V1,PPAN-P2RY11,...,MMP3,CACNA1F,SYN3,FER1L5,FRMD5,ARL13A,SH2D1B,UPK3A,SCN3B,ICAM4
0,TCGA-3C-AALJ,48.459743,0,-728.237,-127.1566,-110.8859,-53.9135,-67.0177,1.3597,0.8971,...,0.2443,-0.1870,0.4740,0.4236,-1.5177,-1.3904,0.2064,2.4360,-0.6435,0.1383
1,TCGA-3C-AALK,47.604958,0,-728.237,-127.1566,-110.8859,-53.9135,-67.0177,1.2012,-0.0455,...,0.6517,-0.6378,0.5622,-0.0767,0.5903,0.0010,-0.8962,-1.4603,0.2582,-0.2475
2,TCGA-4H-AAAK,11.440971,0,-728.237,-127.1566,-110.8859,-53.9135,-67.0177,-0.4894,-0.0184,...,0.3379,-1.2722,0.4053,-0.0564,0.1435,-0.5723,-1.0875,-1.0096,0.8560,0.3781
3,TCGA-5L-AAT0,48.558372,0,-728.237,-127.1566,-110.8859,-53.9135,-67.0177,1.5663,0.6160,...,1.2841,0.7311,-0.6464,0.3602,-0.6255,-0.3252,0.4825,0.2528,0.5292,-0.1505
4,TCGA-5T-A9QA,9.961535,0,-728.237,-127.1566,-110.8859,-53.9135,-67.0177,1.3038,1.0103,...,-2.6086,0.3281,2.8487,-1.4474,0.8469,-0.4196,-1.2204,1.9684,0.1265,-2.1798


In [ ]:
display(data.describe())

,time,event,CEACAM18,KRTAP12-4,RPS4Y2,DEFB134,TTTY4C,TMEM189-UBE2V1,PPAN-P2RY11,ZFP91-CNTF,...,MMP3,CACNA1F,SYN3,FER1L5,FRMD5,ARL13A,SH2D1B,UPK3A,SCN3B,ICAM4
count,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,...,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000,1080.000000
mean,40.705250,0.139815,-726.888413,-126.921125,-110.680556,-53.713820,-66.893593,-1.948713,-0.815471,-1.548750,...,-0.025246,-0.069466,-0.048584,-0.127647,-0.045876,-0.524737,-0.048047,-0.119282,-0.036324,-0.045212
std,39.219216,0.346955,31.323835,5.469582,4.769752,3.277074,2.882967,1.853055,1.718570,1.584666,...,1.039379,1.039295,1.039255,1.039251,1.039242,1.039173,1.039170,1.039141,1.039134,1.039119
min,0.000000,0.000000,-728.237000,-127.156600,-110.885900,-53.913500,-67.017700,-3.480900,-3.493800,-2.870100,...,-3.374100,-1.913400,-2.145700,-1.447400,-2.184700,-1.390400,-2.111200,-1.460300,-2.514600,-2.179800
25%,14.720387,0.000000,-728.237000,-127.156600,-110.885900,-53.913500,-67.017700,-3.480900,-3.375775,-2.870100,...,-0.517750,-0.944600,-0.718300,-0.889875,-0.734875,-1.390400,-0.759675,-0.949800,-0.713450,-0.737775
50%,27.007923,0.000000,-728.237000,-127.156600,-110.885900,-53.913500,-67.017700,-3.480900,-0.125400,-2.870100,...,0.127650,-0.052250,-0.128650,-0.388300,-0.096700,-0.697800,-0.102200,-0.336400,-0.060550,-0.133750
75%,54.730907,0.000000,-728.237000,-127.156600,-110.885900,-53.913500,-67.017700,-0.385550,0.534150,-0.329950,...,0.662575,0.680600,0.497025,0.447975,0.535350,-0.046775,0.542200,0.490775,0.685600,0.580050
max,282.901009,1.000000,1.000000,1.000000,1.000000,1.458000,1.000000,3.753100,1.863900,3.742000,...,2.626600,2.488200,4.538100,3.627700,3.847500,7.091400,5.559200,3.372600,3.812700,3.884300


In [ ]:
# Exclude non-numeric columns and the 'event' column from correlation calculation with 'time'
gene_expression_cols = data.select_dtypes(include=np.number).columns.drop(['time', 'event'], errors='ignore')

# Calculate correlation of each gene with 'time'
correlations = data[gene_expression_cols].corrwith(data['time'])

# Sort correlations by absolute value to find the strongest ones
sorted_correlations = correlations.abs().sort_values(ascending=False)

print("Top 10 genes positively correlated with survival time:")
display(correlations.loc[sorted_correlations.head(10).index].sort_values(ascending=False))

print("\nTop 10 genes negatively correlated with survival time:")
display(correlations.loc[sorted_correlations.head(10).index].sort_values(ascending=True))

Top 10 genes positively correlated with survival time:


,0
PRINS,0.149993
PRO0628,0.128770
PHYHIPL,0.114132
TSLP,0.113679
PAK7,0.113192
EDN3,0.111241
ALX4,0.111182
SLC7A3,0.110609
HAVCR1P1,-0.113820
CEBPE,-0.142159



Top 10 genes negatively correlated with survival time:


,0
CEBPE,-0.142159
HAVCR1P1,-0.113820
SLC7A3,0.110609
ALX4,0.111182
EDN3,0.111241
PAK7,0.113192
TSLP,0.113679
PHYHIPL,0.114132
PRO0628,0.128770
PRINS,0.149993


The results above show the 10 genes with the strongest correlations (both positive and negative) with survival time. Genes with a positive correlation are associated with longer survival time, while those with a negative correlation may indicate shorter survival time.